# ROC-AUC Curve Analysis

Evaluating model performance using ROC curve and AUC metric.

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

print("Libraries imported successfully!")

## Load and Prepare Data

In [ ]:
# Read CSV file
df = pd.read_csv('../data/dataset.csv')

print(f"Dataset shape: {df.shape}")

# Separate features and target
target_col = "Default"
X = df.drop(columns=["LoanID", target_col])
y = df[target_col]

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"Features shape: {X_encoded.shape}")

## Train-Test Split and Scaling

In [ ]:
# Split dataset with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

## Train Logistic Regression Model

In [ ]:
# Train Logistic Regression model with improvements
model_improved = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

model_improved.fit(X_train_scaled, y_train)

print("Model trained successfully!")

## Generate Predictions and Probabilities

In [ ]:
# Get prediction probabilities (important for ROC curve)
y_prob = model_improved.predict_proba(X_test_scaled)[:, 1]

# Get binary predictions
y_pred = model_improved.predict(X_test_scaled)

print(f"Probability predictions shape: {y_prob.shape}")
print(f"Binary predictions shape: {y_pred.shape}")
print(f"\nProbability range: [{y_prob.min():.4f}, {y_prob.max():.4f}]")

## Calculate ROC-AUC Score

In [ ]:
# === ROC-AUC Analysis ===

# Calculate ROC-AUC score
roc_auc = roc_auc_score(y_test, y_prob)

print("="*60)
print("ROC-AUC SCORE")
print("="*60)
print(f"\nROC-AUC Score: {roc_auc:.4f}")
print(f"\nInterpretation:")
print(f"  - 0.5 = Random classifier (useless)")
print(f"  - 0.6-0.7 = Fair")
print(f"  - 0.7-0.8 = Good")
print(f"  - 0.8-0.9 = Very Good")
print(f"  - 0.9-1.0 = Excellent")

if roc_auc >= 0.9:
    quality = "Excellent"
elif roc_auc >= 0.8:
    quality = "Very Good"
elif roc_auc >= 0.7:
    quality = "Good"
elif roc_auc >= 0.6:
    quality = "Fair"
else:
    quality = "Poor"
    
print(f"\nYour model: {quality} ({roc_auc:.4f})")

## Calculate ROC Curve

In [ ]:
# Calculate ROC curve points
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

print(f"\nROC Curve Points:")
print(f"False Positive Rates (FPR) shape: {fpr.shape}")
print(f"True Positive Rates (TPR) shape: {tpr.shape}")
print(f"Thresholds shape: {thresholds.shape}")

print(f"\nFirst few points:")
for i in range(min(5, len(fpr))):
    print(f"  FPR: {fpr[i]:.4f}, TPR: {tpr[i]:.4f}, Threshold: {thresholds[i]:.4f}")

## Plot ROC Curve

In [ ]:
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.4f})', color='blue')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=2, label='Random Classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=12, fontweight='bold')
plt.title('ROC Curve - Logistic Regression Model', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

print("ROC Curve plotted successfully!")

## What is ROC-AUC?

In [ ]:
print("="*60)
print("UNDERSTANDING ROC-AUC")
print("="*60)

print("""
ROC (Receiver Operating Characteristic) Curve:

1. What it measures:
   - How well the model separates defaults from non-defaults
   - Performance across ALL possible classification thresholds

2. Axes:
   - X-axis: False Positive Rate (FPR) = FP / (TN + FP)
     * Proportion of non-defaults incorrectly classified as defaults
     * Range: 0 to 1
   
   - Y-axis: True Positive Rate (TPR) = TP / (TP + FN)
     * Proportion of defaults correctly identified
     * Range: 0 to 1
     * Also called "Recall" or "Sensitivity"

3. AUC (Area Under Curve):
   - Summarizes ROC curve into single number (0 to 1)
   - Probability that model ranks a random default higher than non-default
   
4. Interpretation:
   - AUC = 0.5: Model is useless (same as random guessing)
   - AUC = 1.0: Model is perfect (always correct)
   - Higher AUC = Better model

5. Advantages over Accuracy:
   - Works well with imbalanced data
   - Considers all classification thresholds
   - More reliable metric for binary classification
""")

print("\n" + "="*60)

## Using Different Thresholds

In [ ]:
print("="*60)
print("THRESHOLD ANALYSIS")
print("="*60)

print("""
Default threshold = 0.5:
  - Probability >= 0.5 → Predict Default (1)
  - Probability < 0.5 → Predict Non-Default (0)

What if we want to catch MORE defaults?
  - Lower threshold (e.g., 0.3)
  - More defaults caught (higher recall)
  - More false alarms (lower precision)

What if we want to reduce false alarms?
  - Raise threshold (e.g., 0.7)
  - Fewer false positives (higher precision)
  - More defaults missed (lower recall)

The ROC curve shows this trade-off across all thresholds!
""")

# Show some example thresholds
example_thresholds = [0.3, 0.5, 0.7]

for thresh in example_thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred_thresh)
    tn, fp, fn, tp = cm.ravel()
    fpr_thresh = fp / (fp + tn) if (fp + tn) > 0 else 0
    tpr_thresh = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\nThreshold = {thresh}:")
    print(f"  FPR: {fpr_thresh:.4f}, TPR: {tpr_thresh:.4f}")
    print(f"  TP (Defaults caught): {tp}, FN (Defaults missed): {fn}")
    print(f"  TN (Correct non-defaults): {tn}, FP (False alarms): {fp}")

## Key Takeaways

### ROC-AUC is Superior to Accuracy for:

1. **Imbalanced Datasets**
   - Accuracy is misleading when classes are imbalanced
   - ROC-AUC gives fair assessment

2. **Understanding Trade-offs**
   - Shows recall vs precision trade-off visually
   - Helps choose optimal threshold

3. **Model Comparison**
   - Easy to compare models (single number)
   - Higher AUC = Better discriminative ability

### Business Decision:
- **High recall (catch defaults)**: Lower threshold, accept more false positives
- **High precision (reduce false alarms)**: Raise threshold
- **Balanced**: Use default 0.5 threshold
- **Best for lending**: Prioritize recall (don't miss real defaults)